# AI-Powered Test Oracle: Assertion Generator Training

This notebook trains an AI model to generate intelligent test assertions from API and GraphQL test responses.

**Input**: Test response data from `test_responses.json`  
**Output**: AI-generated assertions for REST API and GraphQL tests

## 1. Setup and Imports

In [ ]:
# Install required packages
!pip install openai anthropic pandas numpy xml.etree.ElementTree lxml python-dotenv tiktoken

In [ ]:
import json
import pandas as pd
import numpy as np
from pathlib import Path
import xml.etree.ElementTree as ET
from datetime import datetime
import re
from typing import List, Dict, Any
import os

# Try to load environment variables (optional)
try:
    from dotenv import load_dotenv
    load_dotenv()
    print("✓ Environment variables loaded from .env file")
except ImportError:
    print("⚠️ python-dotenv not installed. You can set API keys directly in the notebook or install with: pip install python-dotenv")

print("✓ All imports successful")

## 2. Configuration

In [ ]:
# Workspace directory
WORKSPACE_DIR = Path("/Users/jisnyvarghese/a-i-powered-test-oracle-intelligent-assertion-generator-21a31ccd")

# Input files
TEST_RESPONSES_FILE = WORKSPACE_DIR / "test_responses.json"
REST_API_FILE = WORKSPACE_DIR / "rest_api_responses.json"
GRAPHQL_FILE = WORKSPACE_DIR / "graphql_responses.json"

# Output files
ASSERTIONS_OUTPUT = WORKSPACE_DIR / "generated_assertions.json"
TRAINING_REPORT = WORKSPACE_DIR / "training_report.md"

# AI Configuration
AI_PROVIDER = "openai"  # Options: "openai", "anthropic"
MODEL_NAME = "gpt-4"  # or "gpt-3.5-turbo", "claude-3-opus-20240229"

print(f"✓ Configuration set")
print(f"  Workspace: {WORKSPACE_DIR}")
print(f"  AI Provider: {AI_PROVIDER}")
print(f"  Model: {MODEL_NAME}")

### Optional: Set API Keys Directly (if not using .env file)

In [ ]:
# Uncomment and set your API key here if not using .env file
# os.environ["OPENAI_API_KEY"] = "your-openai-api-key-here"
# os.environ["ANTHROPIC_API_KEY"] = "your-anthropic-api-key-here"

# Check if API key is set
if AI_PROVIDER == "openai":
    if os.getenv("OPENAI_API_KEY"):
        print("✓ OpenAI API key found")
    else:
        print("⚠️ OpenAI API key not set. Please set OPENAI_API_KEY environment variable or uncomment the line above.")
elif AI_PROVIDER == "anthropic":
    if os.getenv("ANTHROPIC_API_KEY"):
        print("✓ Anthropic API key found")
    else:
        print("⚠️ Anthropic API key not set. Please set ANTHROPIC_API_KEY environment variable or uncomment the line above.")

## 3. Load Test Data

In [ ]:
def load_json_file(file_path):
    """Load JSON file and return data"""
    if not file_path.exists():
        print(f"⚠️ File not found: {file_path}")
        return None
    
    with open(file_path, 'r', encoding='utf-8') as f:
        data = json.load(f)
    
    print(f"✓ Loaded {len(data)} records from {file_path.name}")
    return data

# Load all test data
print("\n📂 Loading test data...\n")
all_responses = load_json_file(TEST_RESPONSES_FILE)
rest_responses = load_json_file(REST_API_FILE)
graphql_responses = load_json_file(GRAPHQL_FILE)

print(f"\n📊 Data Summary:")
print(f"  Total responses: {len(all_responses) if all_responses else 0}")
print(f"  REST API: {len(rest_responses) if rest_responses else 0}")
print(f"  GraphQL: {len(graphql_responses) if graphql_responses else 0}")

## 4. Parse XML Test Responses

In [ ]:
def parse_xml_response(xml_content):
    """
    Parse XML test response and extract key information
    """
    try:
        root = ET.fromstring(xml_content)
        
        parsed_data = {
            'test_name': root.get('name', 'Unknown'),
            'test_class': root.get('classname', 'Unknown'),
            'time': root.get('time', '0'),
            'status': 'passed',
            'properties': {},
            'system_out': '',
            'system_err': ''
        }
        
        # Check for failures or errors
        failure = root.find('failure')
        error = root.find('error')
        
        if failure is not None:
            parsed_data['status'] = 'failed'
            parsed_data['failure_message'] = failure.get('message', '')
            parsed_data['failure_type'] = failure.get('type', '')
            parsed_data['failure_text'] = failure.text or ''
        
        if error is not None:
            parsed_data['status'] = 'error'
            parsed_data['error_message'] = error.get('message', '')
            parsed_data['error_type'] = error.get('type', '')
            parsed_data['error_text'] = error.text or ''
        
        # Extract properties
        properties = root.find('properties')
        if properties is not None:
            for prop in properties.findall('property'):
                name = prop.get('name', '')
                value = prop.get('value', '')
                parsed_data['properties'][name] = value
        
        # Extract system output
        system_out = root.find('system-out')
        if system_out is not None and system_out.text:
            parsed_data['system_out'] = system_out.text
        
        system_err = root.find('system-err')
        if system_err is not None and system_err.text:
            parsed_data['system_err'] = system_err.text
        
        return parsed_data
    
    except ET.ParseError as e:
        print(f"⚠️ XML Parse Error: {e}")
        return None

# Parse all responses
print("\n🔍 Parsing XML responses...\n")

parsed_responses = []
for response in all_responses:
    content = response.get('content', '')
    if content:
        parsed = parse_xml_response(content)
        if parsed:
            parsed['filename'] = response['filename']
            parsed['api_type'] = response.get('api_type', 'Unknown')
            parsed['url'] = response['url']
            parsed_responses.append(parsed)

print(f"✓ Successfully parsed {len(parsed_responses)} responses")

# Display sample
if parsed_responses:
    print("\n📋 Sample Parsed Response:")
    sample = parsed_responses[0]
    print(json.dumps(sample, indent=2)[:500] + "...")

## 5. Analyze Test Patterns

In [ ]:
def analyze_test_patterns(parsed_responses):
    """
    Analyze patterns in test responses to understand common structures
    """
    analysis = {
        'total_tests': len(parsed_responses),
        'passed': 0,
        'failed': 0,
        'error': 0,
        'rest_api': 0,
        'graphql': 0,
        'common_properties': {},
        'test_classes': set(),
        'avg_execution_time': 0
    }
    
    total_time = 0
    
    for response in parsed_responses:
        # Count status
        status = response.get('status', 'unknown')
        if status == 'passed':
            analysis['passed'] += 1
        elif status == 'failed':
            analysis['failed'] += 1
        elif status == 'error':
            analysis['error'] += 1
        
        # Count API types
        api_type = response.get('api_type', 'Unknown')
        if api_type == 'REST':
            analysis['rest_api'] += 1
        elif api_type == 'GraphQL':
            analysis['graphql'] += 1
        
        # Collect test classes
        test_class = response.get('test_class', 'Unknown')
        analysis['test_classes'].add(test_class)
        
        # Sum execution time
        try:
            time_val = float(response.get('time', 0))
            total_time += time_val
        except:
            pass
        
        # Collect common properties
        properties = response.get('properties', {})
        for key in properties.keys():
            analysis['common_properties'][key] = analysis['common_properties'].get(key, 0) + 1
    
    # Calculate average time
    if analysis['total_tests'] > 0:
        analysis['avg_execution_time'] = total_time / analysis['total_tests']
    
    # Convert set to list for JSON serialization
    analysis['test_classes'] = list(analysis['test_classes'])
    
    return analysis

# Analyze patterns
print("\n📊 Analyzing test patterns...\n")
pattern_analysis = analyze_test_patterns(parsed_responses)

print("Test Pattern Analysis:")
print(f"  Total tests: {pattern_analysis['total_tests']}")
print(f"  Passed: {pattern_analysis['passed']}")
print(f"  Failed: {pattern_analysis['failed']}")
print(f"  Error: {pattern_analysis['error']}")
print(f"  REST API: {pattern_analysis['rest_api']}")
print(f"  GraphQL: {pattern_analysis['graphql']}")
print(f"  Avg execution time: {pattern_analysis['avg_execution_time']:.3f}s")
print(f"  Unique test classes: {len(pattern_analysis['test_classes'])}")

## 6. Initialize AI Model

In [ ]:
def initialize_ai_client(provider="openai"):
    """
    Initialize AI client based on provider
    """
    if provider == "openai":
        try:
            import openai
            api_key = os.getenv("OPENAI_API_KEY")
            if not api_key:
                print("⚠️ OPENAI_API_KEY not found in environment variables")
                print("Please set it in a .env file or environment")
                return None
            
            client = openai.OpenAI(api_key=api_key)
            print("✓ OpenAI client initialized")
            return client
        except ImportError:
            print("❌ OpenAI package not installed. Run: pip install openai")
            return None
    
    elif provider == "anthropic":
        try:
            import anthropic
            api_key = os.getenv("ANTHROPIC_API_KEY")
            if not api_key:
                print("⚠️ ANTHROPIC_API_KEY not found in environment variables")
                return None
            
            client = anthropic.Anthropic(api_key=api_key)
            print("✓ Anthropic client initialized")
            return client
        except ImportError:
            print("❌ Anthropic package not installed. Run: pip install anthropic")
            return None
    
    else:
        print(f"❌ Unknown provider: {provider}")
        return None

# Initialize AI client
print("\n🤖 Initializing AI client...\n")
ai_client = initialize_ai_client(AI_PROVIDER)

if not ai_client:
    print("\n⚠️ AI client not initialized. You can still analyze data without AI generation.")

## 7. Create Assertion Generation Prompts

In [ ]:
def create_assertion_prompt(test_data, api_type="REST"):
    """
    Create a prompt for AI to generate intelligent assertions
    """
    
    base_prompt = f"""
You are an expert test automation engineer specializing in {api_type} API testing.

Given the following test execution data, generate intelligent, comprehensive test assertions that go beyond simple type checking.

Test Data:
- Test Name: {test_data.get('test_name', 'Unknown')}
- Test Class: {test_data.get('test_class', 'Unknown')}
- Status: {test_data.get('status', 'Unknown')}
- Execution Time: {test_data.get('time', '0')}s
- API Type: {api_type}

Properties:
{json.dumps(test_data.get('properties', {}), indent=2)}

System Output:
{test_data.get('system_out', 'No output')[:1000]}

Generate assertions that validate:
1. **Data Types**: Ensure fields have correct types
2. **Business Logic**: Validate domain-specific rules (e.g., IDs are positive, dates are valid)
3. **Relationships**: Check data consistency and relationships
4. **Edge Cases**: Consider boundary conditions
5. **State Transitions**: Validate proper state changes
"""
    
    if api_type == "GraphQL":
        base_prompt += """
6. **Schema Compliance**: Ensure response matches GraphQL schema
7. **Field-Level Validation**: Validate nested objects and arrays
8. **Nullable Fields**: Check nullable vs non-nullable fields
"""
    else:
        base_prompt += """
6. **HTTP Status Codes**: Validate appropriate status codes
7. **Response Headers**: Check required headers
8. **Pagination**: Validate pagination logic if applicable
"""
    
    base_prompt += """

Output Format:
Provide assertions in JSON format with the following structure:
{
  "test_name": "<test_name>",
  "api_type": "<REST|GraphQL>",
  "assertions": [
    {
      "category": "<type|business_logic|relationship|edge_case|state|schema|http>",
      "assertion": "<assertion description>",
      "code": "<code snippet in Kotlin/JavaScript/Python>",
      "priority": "<high|medium|low>"
    }
  ]
}
"""
    
    return base_prompt

# Test prompt creation
if parsed_responses:
    sample_prompt = create_assertion_prompt(parsed_responses[0], parsed_responses[0].get('api_type', 'REST'))
    print("\n📝 Sample Assertion Generation Prompt:\n")
    print(sample_prompt[:800] + "...")

## 8. Generate Assertions with AI

In [ ]:
def generate_assertions_with_ai(test_data, ai_client, provider="openai", model="gpt-4"):
    """
    Generate assertions using AI model
    """
    api_type = test_data.get('api_type', 'REST')
    prompt = create_assertion_prompt(test_data, api_type)
    
    try:
        if provider == "openai":
            response = ai_client.chat.completions.create(
                model=model,
                messages=[
                    {"role": "system", "content": "You are an expert test automation engineer."},
                    {"role": "user", "content": prompt}
                ],
                temperature=0.7,
                max_tokens=2000
            )
            
            content = response.choices[0].message.content
            
            # Try to extract JSON from response
            json_match = re.search(r'\{[\s\S]*\}', content)
            if json_match:
                return json.loads(json_match.group())
            else:
                return {"raw_response": content}
        
        elif provider == "anthropic":
            response = ai_client.messages.create(
                model=model,
                max_tokens=2000,
                messages=[
                    {"role": "user", "content": prompt}
                ]
            )
            
            content = response.content[0].text
            
            # Try to extract JSON from response
            json_match = re.search(r'\{[\s\S]*\}', content)
            if json_match:
                return json.loads(json_match.group())
            else:
                return {"raw_response": content}
    
    except Exception as e:
        print(f"❌ Error generating assertions: {e}")
        return None

# Generate assertions for sample tests
if ai_client and parsed_responses:
    print("\n🤖 Generating AI-powered assertions...\n")
    
    generated_assertions = []
    
    # Generate for first 5 tests as examples
    num_samples = min(5, len(parsed_responses))
    
    for i, test_data in enumerate(parsed_responses[:num_samples]):
        print(f"  [{i+1}/{num_samples}] Generating assertions for: {test_data.get('test_name', 'Unknown')}")
        
        assertions = generate_assertions_with_ai(test_data, ai_client, AI_PROVIDER, MODEL_NAME)
        
        if assertions:
            assertions['source_file'] = test_data.get('filename', 'Unknown')
            assertions['generated_at'] = datetime.now().isoformat()
            generated_assertions.append(assertions)
    
    print(f"\n✓ Generated assertions for {len(generated_assertions)} tests")
    
    # Display sample
    if generated_assertions:
        print("\n📋 Sample Generated Assertions:\n")
        print(json.dumps(generated_assertions[0], indent=2)[:1000] + "...")
else:
    print("\n⚠️ Skipping AI generation (no client or no data)")
    generated_assertions = []

## 9. Save Generated Assertions

In [ ]:
if generated_assertions:
    # Save to JSON
    with open(ASSERTIONS_OUTPUT, 'w', encoding='utf-8') as f:
        json.dump(generated_assertions, f, indent=2, ensure_ascii=False)
    
    print(f"✓ Saved generated assertions to: {ASSERTIONS_OUTPUT}")
    print(f"  Total assertions: {len(generated_assertions)}")
else:
    print("⚠️ No assertions to save")

## 10. Generate Training Report

In [ ]:
def generate_training_report(pattern_analysis, generated_assertions, output_file):
    """
    Generate a comprehensive training report
    """
    report = f"""# AI-Powered Test Oracle: Training Report

**Generated**: {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}

## 📊 Test Data Analysis

### Overall Statistics
- **Total Tests**: {pattern_analysis['total_tests']}
- **Passed**: {pattern_analysis['passed']} ({pattern_analysis['passed']/pattern_analysis['total_tests']*100:.1f}%)
- **Failed**: {pattern_analysis['failed']} ({pattern_analysis['failed']/pattern_analysis['total_tests']*100:.1f}%)
- **Error**: {pattern_analysis['error']} ({pattern_analysis['error']/pattern_analysis['total_tests']*100:.1f}%)

### API Type Distribution
- **REST API**: {pattern_analysis['rest_api']} tests
- **GraphQL**: {pattern_analysis['graphql']} tests

### Performance Metrics
- **Average Execution Time**: {pattern_analysis['avg_execution_time']:.3f}s
- **Unique Test Classes**: {len(pattern_analysis['test_classes'])}

## 🤖 AI-Generated Assertions

- **Total Assertions Generated**: {len(generated_assertions)}
- **AI Model Used**: {MODEL_NAME}
- **Provider**: {AI_PROVIDER}

### Assertion Categories
"""
    
    # Count assertion categories
    category_counts = {}
    for assertion_set in generated_assertions:
        assertions = assertion_set.get('assertions', [])
        for assertion in assertions:
            category = assertion.get('category', 'unknown')
            category_counts[category] = category_counts.get(category, 0) + 1
    
    for category, count in sorted(category_counts.items(), key=lambda x: x[1], reverse=True):
        report += f"- **{category.title()}**: {count} assertions\n"
    
    report += f"""

## 📝 Sample Generated Assertions

"""
    
    # Add sample assertions
    if generated_assertions:
        sample = generated_assertions[0]
        report += f"""### Test: {sample.get('test_name', 'Unknown')}
**API Type**: {sample.get('api_type', 'Unknown')}

"""
        
        assertions = sample.get('assertions', [])[:3]  # First 3 assertions
        for i, assertion in enumerate(assertions, 1):
            report += f"""#### Assertion {i}: {assertion.get('category', 'Unknown').title()}
**Priority**: {assertion.get('priority', 'medium')}

{assertion.get('assertion', 'No description')}

```kotlin
{assertion.get('code', '// No code provided')}
```

"""
    
    report += f"""
## 🎯 Next Steps

1. **Review Generated Assertions**: Examine the assertions in `generated_assertions.json`
2. **Integrate with Tests**: Add assertions to your test framework
3. **Customize Rules**: Adjust AI prompts for domain-specific logic
4. **Continuous Learning**: Feed test results back to improve assertion quality

## 📁 Output Files

- `generated_assertions.json` - All AI-generated assertions
- `training_report.md` - This report
- `test_responses.json` - Original test data

---

*Generated by AI-Powered Test Oracle*
"""
    
    # Save report
    with open(output_file, 'w', encoding='utf-8') as f:
        f.write(report)
    
    print(f"✓ Training report saved to: {output_file}")
    return report

# Generate report
print("\n📄 Generating training report...\n")
report = generate_training_report(pattern_analysis, generated_assertions, TRAINING_REPORT)

# Display report preview
print("\n📋 Report Preview:\n")
print(report[:1000] + "...")

## 11. Visualize Results

In [ ]:
# Create summary DataFrame
if generated_assertions:
    summary_data = []
    
    for assertion_set in generated_assertions:
        test_name = assertion_set.get('test_name', 'Unknown')
        api_type = assertion_set.get('api_type', 'Unknown')
        num_assertions = len(assertion_set.get('assertions', []))
        
        summary_data.append({
            'Test Name': test_name,
            'API Type': api_type,
            'Assertions Generated': num_assertions,
            'Source File': assertion_set.get('source_file', 'Unknown')
        })
    
    df_summary = pd.DataFrame(summary_data)
    
    print("\n📊 Assertion Generation Summary:\n")
    display(df_summary)
    
    print(f"\n📈 Statistics:")
    print(f"  Total assertions: {df_summary['Assertions Generated'].sum()}")
    print(f"  Average per test: {df_summary['Assertions Generated'].mean():.1f}")
    print(f"  Max assertions: {df_summary['Assertions Generated'].max()}")
    print(f"  Min assertions: {df_summary['Assertions Generated'].min()}")
else:
    print("⚠️ No assertions generated to visualize")

## Summary

This notebook provides:

✅ **Data Loading**: Loads test responses from JSON files  
✅ **XML Parsing**: Extracts test information from XML responses  
✅ **Pattern Analysis**: Identifies common patterns in test data  
✅ **AI Integration**: Uses GPT-4 or Claude to generate assertions  
✅ **Intelligent Assertions**: Creates context-aware, business-logic validations  
✅ **Report Generation**: Produces comprehensive training reports  

**Output Files**:
- `generated_assertions.json` - AI-generated assertions
- `training_report.md` - Detailed training report

**Next Steps**:
1. Review generated assertions
2. Integrate with your test framework
3. Customize for your domain
4. Scale to all test cases